# Validation Strategies

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/model-evaluation/02-validation-strategies

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['grid.color']       = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## 1. k-Fold CV Bias-Variance vs k

As k increases, each fold trains on more data (lower bias) but the k estimates become more correlated (higher variance of the CV estimate).

In [ ]:
# Generate a small dataset
n = 200
X = rng.standard_normal((n, 5))
y = (X[:, 0] + 0.5 * X[:, 1] + rng.standard_normal(n) > 0).astype(int)

model = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=300))])

k_values = [2, 3, 5, 10, 20]
means, stds = [], []

for k in k_values:
    scores = cross_val_score(model, X, y, cv=StratifiedKFold(k, shuffle=True, random_state=0))
    means.append(scores.mean())
    stds.append(scores.std())

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(k_values, means, yerr=stds, fmt='o-', color=BRAND, capsize=5,
            linewidth=2, markersize=7, label='Mean CV accuracy ± std')
ax.fill_between(k_values,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.15, color=BRAND)
ax.set_xlabel('k (number of folds)')
ax.set_ylabel('Accuracy')
ax.set_title('k-fold CV: mean accuracy and variance vs k', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for k, m, s in zip(k_values, means, stds):
    print(f'k={k:>2}: mean={m:.4f}  std={s:.4f}')

## 2. Bootstrap OOB Fraction

The fraction of samples *not* selected in a bootstrap sample converges to $1/e \approx 0.368$.

In [ ]:
n_samples = 500
n_bootstrap = 2000
oob_fractions = []

for _ in range(n_bootstrap):
    boot_idx = rng.integers(0, n_samples, n_samples)  # sample with replacement
    oob      = len(set(range(n_samples)) - set(boot_idx)) / n_samples
    oob_fractions.append(oob)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(oob_fractions, bins=40, color=BRAND, alpha=0.7, density=True, edgecolor='none')
ax.axvline(np.mean(oob_fractions), color=TEAL, linewidth=2,
           label=f'Empirical mean = {np.mean(oob_fractions):.4f}')
ax.axvline(1/np.e, color=YELLOW, linewidth=2, linestyle='--',
           label=f'Theoretical 1/e = {1/np.e:.4f}')
ax.set_xlabel('OOB fraction')
ax.set_ylabel('Density')
ax.set_title(f'Bootstrap OOB fraction (n={n_samples}, {n_bootstrap} trials)', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Data Leakage Demo

Fitting a scaler on the entire dataset (before splitting) inflates CV accuracy.
Wrapping in a `Pipeline` fixes this automatically.

In [ ]:
n_leak = 300
X_leak = rng.standard_normal((n_leak, 10))
# Hard problem: only first 2 features matter, rest is noise
y_leak = (X_leak[:, 0] - X_leak[:, 1] > 0).astype(int)

# LEAKY: fit scaler on all data before CV
scaler_leaky = StandardScaler()
X_leak_scaled = scaler_leaky.fit_transform(X_leak)  # sees ALL data including val!
clf = LogisticRegression(max_iter=300)
scores_leaky = cross_val_score(clf, X_leak_scaled, y_leak, cv=5)

# CORRECT: scaler inside pipeline, fit per fold
pipe_correct = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=300))])
scores_correct = cross_val_score(pipe_correct, X_leak, y_leak, cv=5)

print(f'Leaky   CV accuracy: {scores_leaky.mean():.4f} ± {scores_leaky.std():.4f}')
print(f'Correct CV accuracy: {scores_correct.mean():.4f} ± {scores_correct.std():.4f}')
print()
if scores_leaky.mean() > scores_correct.mean():
    print('Leaky pipeline reports HIGHER accuracy — this is the data leakage effect.')
else:
    print('Similar scores — try with a larger, more complex dataset to see the gap.')

## 4. Time-Series Cross-Validation

Walk-forward validation: each split trains on all past data and tests on the next window.

In [ ]:
T = 100
n_splits = 5
step = T // (n_splits + 1)

fig, ax = plt.subplots(figsize=(11, 4))

for i in range(n_splits):
    train_end  = step * (i + 1)
    val_start  = train_end
    val_end    = train_end + step

    ax.barh(i, train_end, left=0, color=BRAND, alpha=0.7, height=0.6)
    ax.barh(i, step, left=val_start, color=ROSE, alpha=0.8, height=0.6)
    ax.text(train_end / 2, i, 'Train', ha='center', va='center', color='white', fontsize=9)
    ax.text(val_start + step / 2, i, 'Val', ha='center', va='center', color='white', fontsize=9)

train_patch = mpatches.Patch(color=BRAND, alpha=0.7, label='Train (expanding)')
val_patch   = mpatches.Patch(color=ROSE,  alpha=0.8, label='Validation')
ax.legend(handles=[train_patch, val_patch])
ax.set_xlabel('Time step')
ax.set_yticks(range(n_splits))
ax.set_yticklabels([f'Split {i+1}' for i in range(n_splits)])
ax.set_title('Walk-forward (expanding window) time-series CV', color='white')
ax.set_xlim(0, T)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## ✏️ Your turn

### Exercise 1 — `kfold_cv_score(X, y, model, k=5)`

Implement k-fold cross-validation from scratch.

1. Split the dataset into k folds (you can use integer division; drop any remainder)
2. For each fold i: train on all other folds, evaluate on fold i
3. Return an array of k validation accuracy scores

**Hint:** `np.array_split` creates k roughly equal parts.

In [ ]:
def kfold_cv_score(X, y, model, k=5):
    """
    k-fold CV from scratch.
    Returns array of k validation accuracy scores.
    """
    X, y = np.asarray(X), np.asarray(y)
    # TODO(you): split into k folds, train on k-1, evaluate on 1
    # Hint:
    #   idx = np.arange(len(X))
    #   folds = np.array_split(idx, k)
    #   for each fold: train_idx = concat of other folds, val_idx = this fold
    scores = []
    return np.array(scores)

In [ ]:
from sklearn.linear_model import LogisticRegression as LR
from sklearn.model_selection import cross_val_score as sk_cvs

clf_test = LR(max_iter=300, random_state=0)
my_scores = kfold_cv_score(X, y, clf_test, k=5)

assert len(my_scores) == 5, f"Expected 5 scores, got {len(my_scores)}"
assert all(0 <= s <= 1 for s in my_scores), "Scores should be in [0, 1]"

# Compare to sklearn's CV (approx — different fold assignment due to no shuffle)
sk_scores = sk_cvs(LR(max_iter=300), X, y, cv=5)
assert abs(my_scores.mean() - sk_scores.mean()) < 0.10, \
    f"Means differ too much: yours={my_scores.mean():.4f}, sklearn={sk_scores.mean():.4f}"

print(f"Your scores:   {my_scores.round(4)}  mean={my_scores.mean():.4f}")
print(f"sklearn scores: {sk_scores.round(4)}  mean={sk_scores.mean():.4f}")
print("\u2705 Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def kfold_cv_score(X, y, model, k=5):
    from sklearn.base import clone
    X, y = np.asarray(X), np.asarray(y)
    idx   = np.arange(len(X))
    folds = np.array_split(idx, k)
    scores = []
    for i in range(k):
        val_idx   = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        m = clone(model)
        m.fit(X[train_idx], y[train_idx])
        acc = np.mean(m.predict(X[val_idx]) == y[val_idx])
        scores.append(acc)
    return np.array(scores)
```

</details>

### Exercise 2 — `bootstrap_oob_fraction(n, n_bootstrap=10000, seed=42)`

Simulate bootstrap sampling to estimate the OOB fraction.

For each of `n_bootstrap` trials:
1. Draw `n` samples with replacement from `{0, 1, ..., n-1}`
2. Count how many original indices were NOT selected

Return the average OOB fraction across all trials.

In [ ]:
def bootstrap_oob_fraction(n, n_bootstrap=10000, seed=42):
    """
    Estimate the fraction of samples not included in a bootstrap sample.
    Returns scalar ~ 1/e ≈ 0.368.
    """
    rng_b = np.random.default_rng(seed)
    # TODO(you): for each bootstrap trial, draw n samples with replacement,
    # compute the OOB fraction, and return the mean
    fractions = []
    return np.mean(fractions)

In [ ]:
oob = bootstrap_oob_fraction(n=500, n_bootstrap=5000, seed=42)
assert oob is not None, "returned None"
assert abs(oob - 1/np.e) < 0.01, \
    f"Expected OOB fraction ~{1/np.e:.4f}, got {oob:.4f}"
print(f"OOB fraction: {oob:.4f}  (theoretical 1/e = {1/np.e:.4f})")
print("\u2705 Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bootstrap_oob_fraction(n, n_bootstrap=10000, seed=42):
    rng_b = np.random.default_rng(seed)
    fractions = []
    for _ in range(n_bootstrap):
        boot = rng_b.integers(0, n, n)
        oob  = len(set(range(n)) - set(boot)) / n
        fractions.append(oob)
    return float(np.mean(fractions))
```

</details>